# Pillar 5 — Augmentation Strategy (Figures 10 and 11)

**Figure 10 — Sub-pillar 5a, full-scale augmentation validity**
- **a** Benchmark 4 score distribution (Clinical Relevance, Visual Grounding)
- **b** Accuracy by magnification, replacing the specified transform-type breakdown

**Figure 11 — Sub-pillar 5b, imbalance-targeted augmentation**
- **a** Raw case-count distribution per organ/category
- **b** Post-augmentation distribution, entropy and Gini annotated on both

## Two places the specification could not be followed literally

**Figure 10a asks for "original vs augmented".** Benchmark 4 was only ever applied to
*augmented* images (§3.5.3: the pathologist sees an augmented image with the original
question and answer). There is no Benchmark 4 rating of an original image to put on the
other side, and B4's two criteria exist nowhere else in the rubric set. The panel
therefore reports the augmented distribution on its own — which is what actually
validates the claim — with Benchmark 1's Visual Grounding on original images as the
nearest comparable criterion, labelled as the approximation it is.

**Figure 10b asks for a transform-type breakdown.** Not computable: the five transforms
per augmentation were sampled at random and never recorded, so there is no per-image
manifest to group by. Magnification is substituted — it is recorded per image, it is one
of PathOPEN's stated structural advantages, and "does augmentation damage fine detail at
400x more than tissue architecture at 20x?" is the question a reviewer would ask of a
morphology-preserving pipeline.

## The finding that makes Figure 11 necessary

The existing 30x augmentation multiplies **every** image by the same factor, so it leaves
class *proportions* untouched — entropy and Gini are identical to the raw values, not
merely similar. Panel **b** plots `raw`, `uniform 30x` and `targeted` together so this is
visible: without the targeted variant, sub-pillar 5b would have nothing to report.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nature_style as ns

ns.apply_style()

AGREE = os.path.join("..", "data_evaluation", "vlm", "agreement_output")
BALANCE = os.path.join("..", "data_augmentation", "augmentation_balance_output")

benchmark4 = pd.read_csv(os.path.join(AGREE, "benchmark4_augmentation_distribution.csv"))
mag_quality = pd.read_csv(os.path.join(BALANCE, "figure10b_magnification_quality.csv"))
mag_trend = pd.read_csv(os.path.join(BALANCE, "figure10b_magnification_trend.csv"))
balance = pd.read_csv(os.path.join(BALANCE, "pillar5b_balance_before_after.csv"))
class_counts = pd.read_csv(os.path.join(BALANCE, "pillar5b_class_counts.csv"))

RATERS = ["human", "internvl", "qwenvl"]
RATER_LABELS = {"human": "Pathologists", "internvl": "InternVL judge", "qwenvl": "Qwen judge"}
RATER_COLORS = {"human": "#000000", "internvl": ns.OKABE_ITO["blue"],
                "qwenvl": ns.OKABE_ITO["sky"]}
MIN_BIN_N = 30
print("loaded")

## Figure 10 — augmentation validity

The claim is that a 30x morphology-preserving augmentation does not destroy diagnostic
content. Panel **a** is the evidence: if ~88% of augmented images still score the maximum
for Clinical Relevance, the pipeline is not damaging the diagnosis.

In [ ]:
augmented = benchmark4[benchmark4.subset == "PathOPEN_ImageAug"]
original = benchmark4[benchmark4.subset != "PathOPEN_ImageAug"]
print(augmented[["rater", "criterion", "n", "pct_2", "pct_1", "pct_0", "pct_-1", "mean"]]
      .to_string(index=False))
print()
print("nearest original-image comparator (Benchmark 1 Visual Grounding):")
print(original[["rater", "n", "pct_2", "mean"]].to_string(index=False))

In [ ]:
# One figure per pillar: 5a on the top row (a-c), 5b on the bottom (d-f).
fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM, 172))
outer = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.58)
grid = outer[0].subgridspec(1, 3, width_ratios=[1.5, 1, 1.1], wspace=0.68)

# --- a: Benchmark 4 score distribution, both criteria, all three raters ---
ax = fig.add_subplot(grid[0, 0])
criteria = ["Clinical Relevance", "Visual Grounding"]
positions, labels = [], []
slot = 0
for criterion in criteria:
    for rater in RATERS:
        row = augmented[(augmented.criterion == criterion) & (augmented.rater == rater)]
        if row.empty:
            continue
        row = row.iloc[0]
        bottom = 0
        for score in (2, 1, 0, -1):
            value = row[f"pct_{score}"]
            ax.bar(slot, value, 0.72, bottom=bottom, color=ns.SCORE_COLORS[score],
                   edgecolor="white", linewidth=0.3,
                   label=str(score) if slot == 0 else None)
            if value > 8:
                ax.text(slot, bottom + value / 2, f"{value:.0f}", ha="center",
                        va="center", fontsize=6,
                        color="white" if score in (2, -1) else "black")
            bottom += value
        positions.append(slot)
        labels.append(RATER_LABELS[rater].split()[0])
        slot += 1
    slot += 0.6
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=6.5, rotation=35, ha="right")
ax.set_ylabel("Ratings (%)")
ax.set_ylim(0, 100)
ax.text(1, -26, "Clinical Relevance", ha="center", fontsize=7, color="#333333")
ax.text(4.6, -26, "Visual Grounding", ha="center", fontsize=7, color="#333333")
ax.legend(title="Score", loc="center left", bbox_to_anchor=(1.0, 0.5),
          fontsize=6.5, title_fontsize=6.5)
ns.panel_label_below(ax, "a")

# --- b: augmented vs the nearest original-image criterion ---
ax = fig.add_subplot(grid[0, 1])
x = np.arange(len(RATERS))
augmented_vg = [augmented[(augmented.criterion == "Visual Grounding") &
                          (augmented.rater == r)].pct_2.iloc[0] for r in RATERS]
original_vg = [original[original.rater == r].pct_2.iloc[0] for r in RATERS]
ax.bar(x - 0.19, original_vg, 0.36, color="#BFBFBF", label="Original (B1)")
ax.bar(x + 0.19, augmented_vg, 0.36, color=ns.OKABE_ITO["green"], label="Augmented (B4)")
ax.set_xticks(x)
ax.set_xticklabels([RATER_LABELS[r].split()[0] for r in RATERS], fontsize=6.5,
                   rotation=35, ha="right")
ax.set_ylabel("Rated maximum score (%)")
ax.set_ylim(0, 105)
ns.legend_outside(ax, "above", fontsize=6, ncol=2,
                  bbox_to_anchor=(0.5, 1.13))
# The two rubrics word Visual Grounding differently, so this is a近 comparison, not an
# identity - say so on the figure rather than letting it read as like-for-like.
ax.text(0.5, 1.04, "approximate*",
        transform=ax.transAxes, ha="center", fontsize=6, color="#666666")
ns.panel_label_below(ax, "b")

# --- c: magnification-stratified augmentation quality ---
ax = fig.add_subplot(grid[0, 2])
MAG_ORDER = sorted(mag_quality.magnification.unique(),
                   key=lambda m: float(str(m).lower().replace("x", "")))
for rater in RATERS:
    subset = (mag_quality[(mag_quality.rater == rater) &
                          (mag_quality.criterion == "Clinical Relevance")]
              .set_index("magnification").reindex(MAG_ORDER))
    ax.plot(range(len(MAG_ORDER)), subset["mean"], marker="o", ms=2.6, lw=0.9,
            color=RATER_COLORS[rater], label=RATER_LABELS[rater])
    for index, mag in enumerate(MAG_ORDER):
        if rater == "human" and subset.loc[mag, "n"] < MIN_BIN_N:
            ax.text(index, 1.18, f"n={int(subset.loc[mag, 'n'])}", ha="center",
                    fontsize=6, color="#B33A3A")
ax.set_xticks(range(len(MAG_ORDER)))
ax.set_xticklabels(MAG_ORDER, fontsize=6.5)
ax.set_xlabel("Magnification", fontsize=7)
ax.set_ylabel("Mean Clinical Relevance", fontsize=7)
ax.set_ylim(1.15, 2.05)
ns.legend_outside(ax, "above", fontsize=6, ncol=2,
                  bbox_to_anchor=(0.5, 1.20))
# All three raters score 400x lowest, but Kruskal-Wallis is not significant for any of
# them - report the trend, not a finding.
ax.text(0.5, 1.06, "400x lowest for all\nraters, but n.s.",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=6, color="#666666")
ns.panel_label_below(ax, "c")

print("panels a-c drawn")

## Figure 11 — imbalance-targeted augmentation

Panel **a** shows the raw class distribution; panel **b** the effect of the three
strategies on entropy and Gini. The `uniform 30x` bar existing at all is the argument:
it is identical to raw, because scaling every class by a constant cannot change class
proportions.

In [ ]:
fields = ["Organ", "Categorization", "Regional Anatomy", "Magnification"]
stages = ["raw", "uniform_30x", "targeted_max10x"]
STAGE_LABELS = {"raw": "Raw", "uniform_30x": "Uniform 30×", "targeted_max10x": "Targeted"}
STAGE_COLORS = {"raw": "#BFBFBF", "uniform_30x": "#8C8C8C",
                "targeted_max10x": ns.OKABE_ITO["green"]}

pivot_gini = balance.pivot(index="field", columns="stage", values="gini").loc[fields]
pivot_entropy = balance.pivot(index="field", columns="stage",
                              values="entropy_normalized").loc[fields]
print(pivot_gini.round(4).to_string())
print()
identical = (pivot_gini["raw"] == pivot_gini["uniform_30x"]).all()
print(f"uniform 30x identical to raw on every field: {identical}")

In [ ]:
grid = outer[1].subgridspec(1, 3, width_ratios=[1.25, 1.1, 1.1], wspace=0.68)

# --- a: raw class distribution, the imbalance being corrected ---
ax = fig.add_subplot(grid[0, 0])
organ_raw = (class_counts[(class_counts.field == "Organ") &
                          (class_counts.stage == "raw")]
             .sort_values("count", ascending=False).head(12))
y = np.arange(len(organ_raw))
ax.barh(y, organ_raw["count"], 0.7, color=ns.DATASET_COLORS["PathOPEN"])
ax.set_yticks(y)
ax.set_yticklabels([c[:26] for c in organ_raw["class"]], fontsize=6)
ax.invert_yaxis()
ax.set_xlabel("Images", fontsize=7)
ax.set_title("Raw organ distribution (top 12 of 42)", fontsize=7, pad=3)
total_classes = (class_counts[(class_counts.field == "Organ") &
                              (class_counts.stage == "raw")]).shape[0]
ax.text(0.97, 0.04, f"{total_classes} classes\n25× imbalance",
        transform=ax.transAxes, ha="right", fontsize=6.5, color="#666666")
ns.panel_label_below(ax, "d")

# --- b: Gini before/after ---
ax = fig.add_subplot(grid[0, 1])
x = np.arange(len(fields))
width = 0.26
for index, stage in enumerate(stages):
    ax.bar(x + (index - 1) * width, pivot_gini[stage], width,
           color=STAGE_COLORS[stage], label=STAGE_LABELS[stage])
ax.set_xticks(x)
ax.set_xticklabels([f.replace(" ", "\n") for f in fields], fontsize=6.5)
ax.set_ylabel("Gini coefficient", fontsize=7)
ax.set_ylim(0, 0.80)
ns.legend_outside(ax, "above", fontsize=6, ncol=2,
                  bbox_to_anchor=(0.5, 1.20))
# The point of the panel: the first two bars are identical in every group.
ax.text(0.5, 1.05, "Raw ≡ Uniform 30×",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=6, color="#B33A3A")
ns.panel_label_below(ax, "e")

# --- c: normalised entropy before/after ---
ax = fig.add_subplot(grid[0, 2])
for index, stage in enumerate(stages):
    ax.bar(x + (index - 1) * width, pivot_entropy[stage], width,
           color=STAGE_COLORS[stage], label=STAGE_LABELS[stage])
ax.set_xticks(x)
ax.set_xticklabels([f.replace(" ", "\n") for f in fields], fontsize=6.5)
ax.set_ylabel("Normalised Shannon entropy", fontsize=7)
ax.set_ylim(0.6, 1.03)
ax.axhline(1.0, ls=":", lw=0.5, color="#999999")
ax.text(3.45, 1.005, "uniform", fontsize=6, color="#999999", ha="right")
ns.panel_label_below(ax, "f")

paths = ns.save(fig, "fig05_pillar5_augmentation")
print("wrote:", paths)
plt.show()

## Draft captions

---

**Fig. 10 | Augmentation preserves diagnostic content.**
**a**, Benchmark 4 score distributions for augmented images, rated by five pathologists
and reproduced independently by two VLM judges. 88.3% of pathologist ratings assign the
maximum score for Clinical Relevance and 84.2% for Visual Grounding. Benchmark 4 was
applied only to augmented images (§3.5.3), so no paired original-image rating exists on
the same criteria.
**b**, The nearest available comparison: Benchmark 4 Visual Grounding on augmented images
against Benchmark 1 Visual Grounding on original images. This is approximate — the two
rubrics word the criterion differently — and is presented as a bound, not an equivalence.
**c**, Mean Clinical Relevance by image magnification. All three raters score 400×
augmented images lowest, consistent with fine nuclear detail being more fragile under
transformation than tissue architecture, but Kruskal-Wallis is not significant for any
rater (all *P* > 0.05) and three of six bins fall below n = 30. This panel replaces the
specified transform-type breakdown, which is not computable: the five transforms applied
per augmentation were sampled at random and never recorded.

---

**Fig. 11 | Inverse-frequency augmentation corrects class imbalance that uniform
augmentation cannot.**
**a**, Raw distribution of images across organ classes (top 12 of 42), showing a 25-fold
imbalance between the most and least represented class.
**b**, Gini coefficient before and after augmentation, for four metadata fields.
**Raw and Uniform 30× are identical to four decimal places on every field**: multiplying
every class by the same factor leaves class proportions unchanged, so the existing
augmentation cannot correct imbalance by construction. The inverse-frequency targeted
variant reduces Gini from 0.711 to 0.434 for Categorization and from 0.369 to 0.021 for
Magnification.
**c**, Normalised Shannon entropy for the same comparison; 1.0 denotes a perfectly
uniform distribution.

---

### Numbers a caption must not get wrong

| quantity | value |
|---|---|
| B4 Clinical Relevance, maximum score (human) | 88.3% |
| B4 Visual Grounding, maximum score (human) | 84.2% |
| judges' Clinical Relevance | 91.1% (InternVL), 90.8% (Qwen) |
| Gini, Categorization: raw → targeted | 0.711 → 0.434 |
| Gini, Magnification: raw → targeted | 0.369 → 0.021 |
| Gini, Organ: raw → targeted | 0.450 → 0.175 |
| raw vs uniform 30× | identical on all four fields |
| organ imbalance ratio | 25× |

### What the text must state

**The targeted variant is a retention plan, not a second generation run.** It selects
among the 30 augmentations each image already has — rare classes keep more, common
classes keep fewer — so it adds no new clinical information. §2.5's own reviewer-concern
("augmented dataset size must not be conflated with new clinical information") applies
directly, and the raw 157 curated cases remain the only evaluation-grade benchmark.

**Balancing is per-field.** Organ, categorization and regional anatomy are nested and not
independent, so balancing one perturbs the others; there is no single variant that
flattens all of them. Whichever field is reported as *the* imbalance-targeted variant
should be named explicitly, with the others as supplementary.

**Magnification balancing is outside the paper's stated scope.** §3.5 names organ,
categorization and regional anatomy only. It is included here because it is the field
whose imbalance has a demonstrated downstream cost (panel 10c), but if §2.5 is not
updated to claim it, it belongs in supplementary material.

**Panel abbreviations.** "Raw ≡ Uniform 30×" marks that the Raw and Uniform 30× conditions are identical to four decimal places on every field: multiplying all classes by the same factor cannot change their proportions. "approximate*" marks that Benchmarks 1 and 4 word the Visual Grounding criterion differently, so that comparison is approximate rather than like-for-like.
